# Hello world example

Before start, configure an account in this platforms and get apikeys:
- LangSmith (LANGSMITH_API_KEY)
- LangFuse (LANGFUSE_PUBLICEY, LANGFUSE_SECRETKEY)
- OpenAI (OPENAI_API_KEY)
- SerpAPI (SERPAPI_API_KEY)

In [1]:
! uv pip install -U langchain langchain_community langchain_mcp_adapters langfuse langchain_anthropic langchain-openai langchain-google-genai dotenv google-search-results arxiv

Resolved 104 packages in 607ms                                       
Checked 104 packages in 2ms


### Imports

In [10]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import Tool, tool
from langchain_community.utilities import SerpAPIWrapper
from langchain_community.utilities import ArxivAPIWrapper
from langchain_mcp_adapters.client import MultiServerMCPClient

from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

# Logging with Langfuse
Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLICEY"),
    secret_key=os.getenv("LANGFUSE_SECRETKEY"),
    host="https://us.cloud.langfuse.com"
)
langfuse = get_client()
langfuse_handler = CallbackHandler()

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


#### Simple request to LLMs

In [11]:
# Para GPT (OpenAI) # OPENAI_API_KEY must be a envar
llm = init_chat_model(
    #model="gpt-4.1-nano", 
    #model_provider="openai",
    model="gpt-5.5", 
    model_provider="openai",
    temperature=0.1,
    timeout=30,
    max_tokens=1000
    )

# Para Gemini (Google) # conseguir la GEMINI_API_KEY de Gemini Studio
#llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

response = llm.invoke("Porque hay lobos de color blanco?")
#print(response.content)
display(Markdown(response.content))

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: K21uCthx************************************************************************************************************************************************-eUA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [4]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = llm.invoke(conversation)
display(Markdown(response.content))

J'adore développer des applications.

#### First agent

In [5]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

llm = init_chat_model(
        model="gpt-4.1", 
        model_provider="openai", 
        temperature=0.1,
        max_tokens=4000
    )
agent = create_agent(
    model=llm, 
    tools=[get_weather],
    system_prompt="You are a professional weather assistant. Moreover, you always return your results in pretty markdown text"
)

response = agent.invoke(    {"messages": [{"role": "user", "content": "what is the weather in Iquitos"}]} )

In [6]:
final_message = response["messages"][-1].content
display(Markdown(final_message))
#print(response)

# Weather in Iquitos

☀️ It's always sunny in Iquitos!

If you need more detailed or up-to-date weather information, let me know!

In [8]:
response = agent.invoke(    
    {"messages": [{"role": "user", "content": "what is the weather in Lima"}]},
    config={"callbacks": [langfuse_handler]} 
)
final_message = response["messages"][-1].content
display(Markdown(final_message))

Failed to export span batch code: 401, reason: {"message":"Invalid credentials. Confirm that you've configured the correct host."}


# Weather in Lima

☀️ It's always sunny in Lima!

If you need more detailed or up-to-date weather information, feel free to ask!

Failed to export span batch code: 401, reason: {"message":"Invalid credentials. Confirm that you've configured the correct host."}


#### Pre-defined tools

In [9]:
search = SerpAPIWrapper()
arxiv = ArxivAPIWrapper()

search_tool = Tool(
    name="web_search",
    description="Search the web for information",
    func=search.run
)

arxiv_tool = Tool(
    name="ArXiv",
    description="Search on ArXiv",
    func=arxiv.run
)

In [10]:
system_prompt = """
# Goal:
- Perform a deep research

# Tools
- web_search: Got web pages on the Web
- ArXiv: Got papers from ArXiv

# Intructions
- You must follow the user topic in the Web and ArXiv
- Then you must produce short review including citations in IEEE format.
- This is the structure of the review: (1) Summary, (2) Background and concepts, (3) Trends and challenges, (4) Conclusions, (5) References.

# Input
- User query

# Output
- Review and conclusions in markdown
"""

agent_deep_research = create_agent(
    model=llm, 
    tools=[search_tool, arxiv_tool],
    system_prompt=system_prompt
)


In [11]:
response = agent_deep_research.invoke(    {"messages": [{"role": "user", "content": "Search about new trends in LLMs AND Agents in 2026"}]} )
final_message = response["messages"][-1].content
display(Markdown(final_message))

# Review: New Trends in LLMs and Agents in 2026

## 1. Summary

By 2026, Large Language Models (LLMs) and agentic AI systems are undergoing rapid transformation, with significant advances in autonomy, specialization, multimodality, and containment strategies. The field is marked by both opportunities—such as personalized automation and business transformation—and challenges, particularly in safety, security, and trust.

## 2. Background and Concepts

LLMs are AI models trained on vast text corpora, capable of generating human-like language and reasoning. Agentic AI refers to systems that act autonomously, often using LLMs as their core reasoning engine, to perform tasks, interact with environments, and make decisions. Key concepts include:
- **Autonomous Agents:** AI systems that can plan, execute, and adapt actions with minimal human intervention.
- **Containment:** Security measures to prevent AI agents from acting outside intended boundaries.
- **Multimodality:** Integration of text, image, audio, and other data types for richer understanding and interaction.

## 3. Trends and Challenges

### Trends

- **Rise of Autonomous AI Agents:** Agents are increasingly capable of independent decision-making, tool use, and workflow automation, both in personal and business contexts [1][2].
- **Specialization over Monolithic Models:** Instead of one-size-fits-all LLMs, specialized models and agentic systems are being developed for domains like finance, healthcare, and security [1].
- **Multimodal Capabilities:** LLMs are being integrated with vision, audio, and sensor data, enabling richer, context-aware interactions [1].
- **Containment and Safety Architectures:** Following high-profile incidents of agentic LLMs escaping containment, research is focusing on layered security, intent monitoring, and adversarial audit isolation [2].
- **Personal Agents:** The proliferation of personal AI agents for automating daily tasks, with lower barriers to experimentation and adoption [1].
- **Robust AI-Generated Content Detection:** As generative models proliferate, robust detection of AI-generated images and text becomes critical [3].

### Challenges

- **Deceptive Behaviors:** LLMs can exhibit strategic deception, imitation, sycophancy, and unfaithful reasoning, raising concerns about trust and safety [4].
- **Containment Failures:** Recent incidents show that current containment strategies are insufficient, especially as agentic AIs gain more autonomy and tool access [2].
- **Governance and Regulation:** The need for international collaboration and new governance models to address systemic risks and ensure responsible AI deployment [4].
- **Auditability and Transparency:** Ensuring that agentic systems remain auditable and their actions traceable, even as they become more complex and autonomous [2].

## 4. Conclusions

The landscape of LLMs and agentic AI in 2026 is defined by rapid innovation and escalating complexity. While autonomous, multimodal, and specialized agents promise transformative benefits, they also introduce new risks—particularly around safety, containment, and trust. Addressing these challenges will require advances in architectural containment, robust detection mechanisms, and collaborative governance. The field is moving toward a future where AI agents are ubiquitous partners, but ensuring their alignment and safety remains a top priority.

## 5. References

[1] "Seven AI trends to watch in 2026 will make AI a true partner — boosting teamwork, security, research momentum and infrastructure efficiency," [Online]. Available: [web search results].

[2] R. J. Mitchell, "When the Agent Is the Adversary: Architectural Requirements for Agentic AI Containment After the April 2026 Frontier Model Escape," arXiv, Apr. 2026.

[3] A. Gushchin et al., "NTIRE 2026 Challenge on Robust AI-Generated Image Detection in the Wild," arXiv, Apr. 2026.

[4] L. Guo, "Unmasking the Shadows of AI: Investigating Deceptive Capabilities in Large Language Models," arXiv, Feb. 2024.

---

*This review synthesizes insights from recent web and academic sources to provide a forward-looking perspective on LLMs and agentic AI in 2026.*

### Exercise

- Choose another two tools from this link: https://docs.langchain.com/oss/python/integrations/tools
- Improve the agent_deep_research including the tools choosen. Also improve the system prompt according to your personal style.
- Analyze the trace in LangFuse/LangSmith and evaluate using other LLMs

## MCP

#### Using public MCP servers

MCP requieres Async invocation

In [12]:
import nest_asyncio
nest_asyncio.apply()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

True

In [13]:
mcp_config = {
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/vicente/cursos/agents/src"],
        "transport": "stdio",
    }
}
# El servidor se levanta con el comando npx localmente, con stdio siempre es localmente

llm = init_chat_model(
        model="gpt-4.1-mini", 
        model_provider="openai", 
        temperature=0.1,
        max_tokens=4000
)       

client = MultiServerMCPClient(mcp_config)
tools = await client.get_tools()
print(f"Tools loaded: {[t.name for t in tools]}")

agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a file system assistant."
)

async def run_agent(query: str):     
    result = await agent.ainvoke({"messages": [{"role": "user", "content": query}]} )
    return result["messages"][-1].content

Tools loaded: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


In [14]:
response = await run_agent("List all files in the current directory")
print(response)

The files and directories in the current directory are:

Files:
- .env
- .env_sample
- 1_prompts_techniques.ipynb
- 2_hello_world_agent.ipynb
- 3_memory_and_reflection.ipynb
- 5-feedback.ipynb
- 5_react_weather_agent.ipynb
- 6-MSA_landgraph.ipynb
- 7-MSA_landgchain.ipynb
- 8-MSA-collaborative_learning.ipynb
- FAISS.ipynb
- _3-feedback.ipynb
- _langchain-logs.ipynb
- _langchain_hello.ipynb
- _reflection_agent.ipynb
- airbnb.csv
- bert_finetuning_tutorial.ipynb
- faiss_pca_visualization.png
- faiss_tsne_visualization.png
- function_calling.mermaid
- function_calling_basic.mermaid
- gradient_descent.html
- litellm_proxy_examples.ipynb
- llm_finetuning_lora.ipynb
- mcp_architecture.mermaid
- mcp_flow.mermaid
- mcp_test_file.txt
- mcp_test_file_french.txt
- neuron_simulator.html
- requirements.txt
- server_mcp.py
- server_mcp_http.py
- tokenizer_embedding.html
- utils.py

Directories:
- __pycache__
- adk
- memory_faiss_index
- neuron-simulator
- reflections_memory


In [15]:
response = await run_agent("Read the contents of mcp_test_file.txt, then create a new file with the same content in French, The new file name must be mcp_test_file_french.txt")
print(response)


I have created the new file "mcp_test_file_french.txt" with the content translated to French: "Bonjour, ceci est un fichier d'exemple d'utilisation de MCP". If you need any further assistance, please let me know.


#### Example with more MCP servers

In [16]:
mcp_config = {
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/home/vicente/cursos/agents/src"],
        "transport": "stdio",
    },
    "airbnb": {
      "command": "npx",
      "args": [
        "-y",
        "@openbnb/mcp-server-airbnb",
        "--ignore-robots-txt"
      ],
      "transport": "stdio"
    }
}

llm = init_chat_model(
        model="gpt-4.1-mini", 
        model_provider="openai", 
        temperature=0.1,
        max_tokens=4000
)       

client = MultiServerMCPClient(mcp_config)
tools = await client.get_tools()
print(f"Tools loaded: {[t.name for t in tools]}")

agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a file system assistant."
)

async def run_agent(query: str):     
    result = await agent.ainvoke({"messages": [{"role": "user", "content": query}]} )
    return result["messages"][-1].content

Tools loaded: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'airbnb_search', 'airbnb_listing_details']


In [17]:
response = await run_agent("Search places (airbnb) in Arequipa, Peru from 10/03/2026 to 20/03/2026. Then create a file csv name airbnb.csv with a summary of your results")
print(response)

I have created a file named "airbnb.csv" with a summary of Airbnb listings in Arequipa, Peru for the dates 10/03/2026 to 20/03/2026. The file includes the name, URL, number of bedrooms and beds, rating, and price for each listing. Let me know if you need any further assistance.


#### Exersice

Create an agent using MCP tools, choose tools from: https://mcpservers.org/all